In [2]:
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from astropy.timeseries import LombScargle
import astropy.units as u
import gyrointerp
from gyrointerp import gyro_age_posterior
from gyrointerp import get_summary_statistics

targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\ASTR502_Mega_Target_List.csv")

In [3]:
#convert txt file to csv
kepler_targets = pd.read_csv("C:\\Users\\smithlt\\Documents\\ASTR502\\Kepler_data.csv")

print(kepler_targets.head())

      TIC ID          2MASS ID  Kepler ID  Kep RA (J2000)  Kep Dec (J2000)  \
0  137220387  19261900+3802089    2853093      291.579170         38.03583   
1  271352188  19395364+4512492    8962094      294.973520         45.21369   
2  158935283  19161861+4600187    9458613      289.077540         46.00522   
3  184009076  19415549+4043551    5546761      295.481229         40.73198   
4  239225129  19472307+4008190    5031857      296.846160         40.13861   

                Condition flag     Period  Period Power  
0                    Exoplanet  26.611830      0.000033  
1                    Exoplanet  41.561185      0.000112  
2                    Exoplanet   2.721708      0.000129  
3  Eclipsing_binary; Exoplanet  11.019765      0.000146  
4          Planetary_candidate   3.542097      0.000147  


In [10]:
#create an array of the Kepler target TIC IDs
kepler_target_tic_ids = kepler_targets['TIC ID'].values
kepler_target_periods = kepler_targets['Period'].values

print(len(kepler_target_tic_ids))

1962


In [23]:
if 'mission_source' in targets.columns:
    print(f"\nMission sources in dataset:")
    print(targets['mission_source'].value_counts())

    # Filter for only Kepler targets
    kepler_targets_mega = targets[targets['mission_source'] == 'Kepler'].copy()
    print(f"\nFound {len(kepler_targets_mega)} Kepler targets!")

    # Clean tic_id: remove leading "TIC " if present and convert to numeric (coerce failures to NA)
    kepler_targets_mega['tic_id'] = kepler_targets_mega['tic_id'].astype(str).str.replace('TIC ', '', regex=False)
    kepler_targets_mega['tic_id'] = pd.to_numeric(kepler_targets_mega['tic_id'], errors='coerce').astype('Int64')
    print(kepler_targets_mega['tic_id'].head())

    # If you have a list/array of TIC IDs to match, filter to those; otherwise keep all Kepler targets
    if 'kepler_target_tic_ids' in globals():
        # ensure kepler_target_tic_ids are integers
        try:
            tic_list = [int(x) for x in kepler_target_tic_ids]
            print(len(tic_list))
        except Exception:
            tic_list = list(kepler_target_tic_ids)
        #should shorten to 1962 targets from the kepler targets csv
        matched = kepler_targets[kepler_targets['TIC ID'].isin(tic_list)].copy()
        matched_teff = kepler_targets_mega[kepler_targets_mega['tic_id'].isin(tic_list)].copy()
        print(len(matched))
        print(f"\nMatched {len(matched)} Kepler targets from provided TIC ID list.")

    # Build kepler_star_df with columns required downstream: 'target_name', 'tic_id', 'Teff', 'period'
    target_results = []
    for idx, r in matched.iterrows():
        tic = int(r['TIC ID']) if pd.notnull(r['TIC ID']) else None
        target_name = r.get('pl_name') or r.get('hostname') or f"TIC{tic}"
        #have to get teff from the mega target list
        teff = matched_teff[matched_teff['tic_id'] == tic]['st_teff'].values
        lit_age = matched_teff[matched_teff['tic_id'] == tic]['st_age'].values

        #want to find the single period value for this tic id
        period = kepler_targets['Period'][kepler_targets['TIC ID'] == tic].values

        target_results.append({
            'tic_ids': target_name,
            'Teff': teff,
            'period': period,
            'st_age': lit_age
        })

    # create DataFrame even if empty so later cells won't raise NameError
    kepler_star_df = pd.DataFrame(target_results, columns=['tic_ids', 'Teff', 'period', 'st_age'])
    print(f"\nFinal kepler_star_df has {len(kepler_star_df)} rows.")

else:
    # If no 'mission_source' column, create empty kepler_star_df to avoid NameError later
    print("No 'mission_source' in targets DataFrame; creating empty kepler_star_df.")
    kepler_star_df = pd.DataFrame(columns=['target_name', 'tic_ids', 'Teff', 'period'])
    print(kepler_star_df.head())


Mission sources in dataset:
mission_source
Kepler    2762
TESS       717
K2         548
WASP       168
HAT        139
Other      105
CoRoT       34
NGTS        22
KELT        21
Name: count, dtype: int64

Found 2762 Kepler targets!
1363    351766445
1364    351766604
1365    351766517
1366    351799800
1369    123126460
Name: tic_id, dtype: Int64
1962
1962

Matched 1962 Kepler targets from provided TIC ID list.

Final kepler_star_df has 1962 rows.


In [25]:
Teff = kepler_star_df['Teff']
print(Teff.head())
Prot = kepler_star_df['period']
print(Prot.head())
lit_age = kepler_star_df['st_age']
print(lit_age.head())

0                                    [5694.0]
1            [5739.0, 5739.0, 5739.0, 5739.0]
2    [5904.0, 5904.0, 5904.0, 5904.0, 5904.0]
3                                    [5683.0]
4                                    [6086.0]
Name: Teff, dtype: object
0    [26.61183003]
1    [41.56118472]
2    [2.721708315]
3    [11.01976462]
4    [3.542096695]
Name: period, dtype: object
0                            [4.07]
1          [1.62, 1.62, 1.62, 1.62]
2    [4.27, 4.27, 4.27, 4.27, 4.27]
3                            [4.17]
4                             [0.3]
Name: st_age, dtype: object


In [26]:
print(kepler_star_df.head())

        tic_ids                                      Teff         period  \
0  TIC137220387                                  [5694.0]  [26.61183003]   
1  TIC271352188          [5739.0, 5739.0, 5739.0, 5739.0]  [41.56118472]   
2  TIC158935283  [5904.0, 5904.0, 5904.0, 5904.0, 5904.0]  [2.721708315]   
3  TIC184009076                                  [5683.0]  [11.01976462]   
4  TIC239225129                                  [6086.0]  [3.542096695]   

                           st_age  
0                          [4.07]  
1        [1.62, 1.62, 1.62, 1.62]  
2  [4.27, 4.27, 4.27, 4.27, 4.27]  
3                          [4.17]  
4                           [0.3]  


In [ ]:
# calculate dictionary of summary statistics for each target and store results in results_df
# Note: gyro_age_posterior and get_summary_statistics were imported in earlier cells,
# so we don't re-import them here.

# ensure columns exist (store arrays as objects)
for col in ['age_grid', 'age_posterior', 'median', '+1sigma', '-1sigma', 'mean', 'mode']:
    if col not in kepler_star_df.columns:
        kepler_star_df[col] = [None] * len(kepler_star_df)

for i in range(kepler_star_df.shape[0]):
    Prot = kepler_star_df['period'].iloc[i]
    Prot_err = 0.2

    Teff = np.mean(kepler_star_df['Teff'].iloc[i])
    Teff_err = 100

    # uniformly spaced grid between 0 and 4000 megayears
    age_grid = np.linspace(0, 4000, 500)

    # calculate the age posterior at each age in `age_grid`
    age_posterior = gyro_age_posterior(
        Prot, Teff,
        Prot_err=Prot_err, Teff_err=Teff_err,
        age_grid=age_grid
    )

    # compute summary statistics
    result = get_summary_statistics(age_grid, age_posterior)

    # store results in the dataframe
    kepler_star_df.at[i, 'age_grid'] = age_grid
    kepler_star_df.at[i, 'age_posterior'] = age_posterior
    kepler_star_df.at[i, 'median'] = result.get('median', np.nan)
    kepler_star_df.at[i, '+1sigma'] = result.get('+1sigma', np.nan)
    kepler_star_df.at[i, '-1sigma'] = result.get('-1sigma', np.nan)
    kepler_star_df.at[i, 'mean'] = result.get('mean', np.nan)
    kepler_star_df.at[i, 'mode'] = result.get('mode', np.nan)

    print(f"\nTarget: {kepler_star_df['tic_ids'].iloc[i]}")
    print(f"Age = {result['median']} +{result['+1sigma']} -{result['-1sigma']} Myr.")
    print(f"Age (most probable): {age_grid[np.argmax(age_posterior)]:.1f} Myr")
